In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sy
import functools as ft
from matplotlib.gridspec import GridSpec

def qubit_eq(o, Te, l): 
    # will be the vectorized equation for a single qubit
    
    # o = time dependent frequency
    # T_e = temperature of the bath 
    # l = coupling to the environment

    no = (np.exp(o/Te) - 1)**(-1)                               # bath is Bosonic
    gp = np.pi * l * o * no                                     # gamma_+
    gm = np.pi * l * o * (1 + no)

    i2 = np.eye(2)
    sp = np.array([[0, 1],[0, 0]], 'complex')
    sm = sp.T
    sz = np.array([[1, 0],[0, -1]], 'complex')

    comm = 0.5 * o * (np.kron(sz, i2) - np.kron(i2, sz.T))
    dm = np.kron(sm, sm) - 0.5 * (np.kron(sp @ sm, i2) + np.kron(i2, sp @ sm))
    dp = np.kron(sp, sp) - 0.5 * (np.kron(sm @ sp, i2) + np.kron(i2, sm @ sp))
    Lou  = -1j * comm + gp * dp + gm * dm
    return Lou

def time_evolve(o, Te, T0, l, t):
    # will evolve the density matrix in time

    dt = t[1] - t[0]                               # time step
    f_T0 = 1/(np.exp(o[0]/T0) + 1)                 # Fermi distribution
    rho_initial = np.array([f_T0, 0, 0, 1-f_T0])   # initial density matrix
    rho_t = np.zeros((4, len(t)))                  # will contain the density matrix in all the steps
    rho_t[:,0] = rho_initial                        

    for i in range(len(t) - 1):                    # loop for time evolution
        rho_t[:, i + 1] = sy.linalg.expm(qubit_eq(o[i + 1], Te, l)*dt) @ rho_t[:, i]
        if np.round(np.trace(rho_t[:, i + 1].reshape(2,2))) != 1:
            raise Exception('Trace not preserved')
        if np.round(np.trace(rho_t[:, i + 1].reshape(2,2) @ rho_t[:, i + 1].reshape(2,2))) > 1:
            raise Exception('Purity not preserved')
    
    return rho_t

def Dissipator(m, rho):
    m_dag = m.conjugate().T
    return m @ rho @ m_dag - 0.5 * (m_dag @ m @ rho + rho @ m_dag @ m)

def commutator(a, b):
    return a @ b - b @ a

def dQdt(o, t, l, Te, rho):
    
    sz = np.array([[1, 0], [0, -1]])
    sm = np.array([[0, 0], [1, 0]])
    heat_curr = np.zeros(len(t))

    for i in range(len(t)) :
        no = (np.exp(o[i]/Te) - 1)**(-1)                               # bath is Bosonic
        gp = np.pi * l * o[i] * no                                     # gamma_+
        gm = np.pi * l * o[i] * (1 + no)

        drho_dt = -1j * 0.5 *  o[i] * (sz @ rho[i] - rho[i] @ sz) + gp * Dissipator(sm.T, rho[i]) + gm * Dissipator(sm, rho[i])
        H_t = 0.5 * o[i] * sz
        heat_curr[i] = np.trace(H_t @ drho_dt)
    
    return heat_curr

def power(o, t, rho):
    # derivative of the frequency because that's the only thing that's time dependent
    dt = t[1] - t[0]
    do_dt = np.gradient(o, dt) 
    sz = np.array([[1, 0],[0, -1]])   
    eye2 = np.eye(2)
    power = np.zeros(len(t))                  # will store the power

    for i in range(len(t)) :
        dH_dt = 0.5 * do_dt[i] * sz   # modified Hamiltonian (sz + I)
        power[i] = np.trace(dH_dt @ rho[i]) + 0.5 * do_dt[i]

    return power

def power_a(o, t, T):
    dt = t[1] - t[0]
    do_dt = np.gradient(o, dt) 
    power = np.zeros(len(t))
    for i in range(len(t)):
        power[i] = do_dt[i]/(np.exp(o[i]/T[i]) + 1)
    return power

def driving_protocol(o, t, Temp, temp_der, env_args, drive_args): 
    # COMPUTES THE DERIVATIVE OF THE DRIVING FREQUENCIES TO BE FED INTO THE SOLVER
    l, Te = env_args
    no = (np.exp(o/Te) - 1)**(-1)                               # bath is Bosonic
    gp = np.pi * l * o * no                                     # gamma_+
    gm = np.pi * l * o * (1 + no)                               # gamma_-
    T = Temp(t, Te, drive_args)                                 # Temperature profile
    dTdt = temp_der(t)                                          # temperature derivative
    dodt = (o/T)*dTdt + T*(np.exp(o/T) + 1)*(gm * np.exp(-o/T) - gp)
    return dodt 


Sine wave implementation.

The qubit is fed a frequency, $\omega(t) = \omega_0 + \epsilon \text{sin}(\alpha t)$.

For the uncoupled case, $\mathcal{T}(t) = \omega(t) T_e$

In [ ]:
Te = 1.5                                # Temperature of the environment
t = np.arange(0, 250, 0.05)             # Time 
freq = 1                                # Omega_0
alpha = 0.1 * freq                      # Sets the frequency of the drive
l1 = 0.001                              # Zero coupling to the environment 
l4 = 5 * 0.001
l2 = 0.00001                            # weak coupling to the environment
l5 = 5 * 0.01
l3 = 0.01                               # somewhat stronger coupling to the environment

o = freq + 0.1 * np.sin(alpha * t)      # Driving in the form of sine pulse

rho_l1 = time_evolve(o, Te, Te, l1, t)      # time_evolution of the density matrix
rho_l2 = time_evolve(o, Te, Te, l2, t)
rho_l3 = time_evolve(o, Te, Te, l3, t)
rho_l4 = time_evolve(o, Te, Te, l4, t)
rho_l5 = time_evolve(o, Te, Te, l5, t)

# Heat current calculation
J_l4_sine= dQdt(o, t, l4, Te, rho_l4.T.reshape((len(t), 2, 2)))
p_l4_sine = power(o, t, rho_l4.T.reshape((len(t), 2, 2)))

p_l4_sine_a = power_a(o, t, o/(freq * np.log((1/rho_l4[0,:]) - 1)))

plt.plot(freq * t, (freq + 0.1 * np.sin(alpha * t))* Te, label = 'Theory')
plt.plot(freq * t, o/(freq * np.log((1/rho_l1[0,:]) - 1)), label = r'$\gamma =$'+str(l1))
plt.plot(freq * t, o/(freq * np.log((1/rho_l2[0,:]) - 1)), label = r'$\gamma = $'+str(l2))
plt.plot(freq * t, o/(freq * np.log((1/rho_l3[0,:]) - 1)), label = r'$\gamma = $'+str(l3))
plt.plot(freq * t, o/(freq * np.log((1/rho_l4[0,:]) - 1)), label = r'$\gamma = $'+str(l4))
plt.plot(freq * t, o/(freq * np.log((1/rho_l5[0,:]) - 1)), label = r'$\gamma = $'+str(l5))
plt.legend()
plt.xlabel(r'$\omega_0 t$')
plt.ylabel(r'$k_B \mathcal{T}(t)/\hbar \omega_0$')

In [ ]:
#heat current and power for the sine drive (lambda = l4)
plt.plot((alpha * t)/(2 * np.pi), J_l4_sine/alpha, label=r'$\mathcal{J}/\hbar\omega_0^2$')
plt.plot((alpha * t)/(2 * np.pi), p_l4_sine/alpha, label=r'$\mathcal{P}/\hbar\omega_0^2$')
plt.xlabel(r'$f t$')
plt.legend()


Sine waves

In [ ]:
# the inverse
def sine_Temp(t, Te, drive_args):
    alpha, freq, amp = drive_args
    return Te * (freq + amp * np.sin(alpha * t))

def sine_dTdt(Te, drive_args, t):
    alpha, freq, amp = drive_args
    return Te * amp * alpha * np.cos(alpha * t)

Te = 1.5
l1 = 0.001                               # Zero coupling to the environment 
l2 = 0.00001                             # weak coupling to the environment
l3 = 0.01                                # somewhat stronger coupling to the environment
drive_args = [0.1, 1, 0.1]
t = np.arange(0, 250, 0.05)

dTdt_sine = ft.partial(sine_dTdt, Te, drive_args)

omi = 1
osine_inv_l1 = sy.integrate.odeint(driving_protocol, omi, t, args=(sine_Temp, dTdt_sine, [l1, Te], drive_args))
osine_inv_l2 = sy.integrate.odeint(driving_protocol, omi, t, args=(sine_Temp, dTdt_sine, [l2, Te], drive_args))
osine_inv_l3 = sy.integrate.odeint(driving_protocol, omi, t, args=(sine_Temp, dTdt_sine, [l3, Te], drive_args))
osine_inv_l4 = sy.integrate.odeint(driving_protocol, omi, t, args=(sine_Temp, dTdt_sine, [l4, Te], drive_args))
osine_inv_l5 = sy.integrate.odeint(driving_protocol, omi, t, args=(sine_Temp, dTdt_sine, [l5, Te], drive_args))

# Heat current calculation
rho_sin_inv_l3 = time_evolve(osine_inv_l3[:,0], Te, sine_Temp(t[0], Te, drive_args), l3, t)
J_l3_sine_inv= dQdt(osine_inv_l3[:,0], t, l3, Te, rho_sin_inv_l3.T.reshape((len(t), 2, 2)))
p_l3_sine_inv = power(osine_inv_l3[:,0], t, rho_sin_inv_l3.T.reshape((len(t), 2, 2)))

plt.plot(t, osine_inv_l1, label = str(l1))
plt.plot(t, osine_inv_l2, label = str(l2))
plt.plot(t, osine_inv_l3, label = str(l3))
plt.plot(t, osine_inv_l4, label = str(l4))
plt.plot(t, osine_inv_l5, label = str(l5))
plt.plot(freq * t, o/(freq * np.log((1/rho_l2[0,:]) - 1)), label = r'$\gamma = $'+str(l2))
plt.plot(t, sine_Temp(t, Te, drive_args), label = 'here')
plt.legend()

In [ ]:
# heat current and power for the sine inverse (lambda = l3)
plt.plot((alpha * t)/(2 * np.pi), J_l3_sine_inv/alpha, label=r'$\mathcal{J}/\hbar\omega_0^2$')
plt.plot((alpha * t)/(2 * np.pi), p_l3_sine_inv/alpha, label=r'$\mathcal{P}/\hbar\omega_0^2$')
plt.xlabel(r'$f t$')
plt.legend()


In [ ]:
# feedback check: feed the inverted frequency back into the Lindblad
# equation and confirm we recover the target temperature profile.
rho_sin_check_l2 = time_evolve(osine_inv_l2[:, 0], Te, sine_Temp(t[0], Te, drive_args), l2, t)
rho_sin_check_l3 = time_evolve(osine_inv_l3[:, 0], Te, sine_Temp(t[0], Te, drive_args), l3, t)
rho_sin_check_l4 = time_evolve(osine_inv_l4[:, 0], Te, sine_Temp(t[0], Te, drive_args), l4, t)

plt.plot(t, sine_Temp(t, Te, drive_args), 'k--', label='target')
plt.plot(t, osine_inv_l2[:, 0]/np.log((1/rho_sin_check_l2[0, :]) - 1), label=r'$\lambda=$'+str(l2))
plt.plot(t, osine_inv_l3[:, 0]/np.log((1/rho_sin_check_l3[0, :]) - 1), label=r'$\lambda=$'+str(l3))
plt.plot(t, osine_inv_l4[:, 0]/np.log((1/rho_sin_check_l4[0, :]) - 1), label=r'$\lambda=$'+str(l4))
plt.xlabel(r'$\omega_0 t$')
plt.ylabel(r'$k_B \mathcal{T}(t)/\hbar\omega_0$')
plt.legend()


Sawtooth pulses

In [ ]:
l1 = 0.001  # zero coupling to the environment 
l2 = 0.00001 # weak coupling to the environment
l3 = 0.01 # somewhat stronger coupling to the environment

omega_0 = 1
t = np.arange(0, 250, 0.05)
phase = np.pi/2
osw = omega_0 + 0.1 * sy.signal.sawtooth(alpha * t + phase, width = 0.5)

rho_l1sw = time_evolve(osw, Te, Te, l1, t)
rho_l2sw = time_evolve(osw, Te, Te, l2, t)
rho_l3sw = time_evolve(osw, Te, Te, l3, t)
rho_l4sw = time_evolve(osw, Te, Te, l4, t)
rho_l5sw = time_evolve(osw, Te, Te, l5, t)

# Heat current calculation
J_l4_sw= dQdt(osw, t, l4, Te, rho_l4sw.T.reshape((len(t), 2, 2)))
p_l4_sw = power(osw, t, rho_l4sw.T.reshape((len(t), 2, 2)))

p_l4_sw_a = power_a(osw, t, osw/(freq * np.log((1/rho_l4sw[0,:]) - 1)))

plt.plot(freq * t, (freq + 0.1 * sy.signal.sawtooth(alpha * t + np.pi/2, 0.5))* Te, label = 'Theory') #need to adjust this
plt.plot(freq * t, osw/(freq * np.log((1/rho_l2sw[0,:]) - 1)), label = r'$\gamma = $'+str(l2))
plt.plot(freq * t, osw/(freq * np.log((1/rho_l3sw[0,:]) - 1)), label = r'$\gamma = $'+str(l3))
plt.plot(freq * t, osw/(freq * np.log((1/rho_l1sw[0,:]) - 1)), label = r'$\gamma = $'+str(l1))
plt.plot(freq * t, osw/(freq * np.log((1/rho_l4sw[0,:]) - 1)), label = r'$\gamma = $'+str(l4))
plt.plot(freq * t, osw/(freq * np.log((1/rho_l5sw[0,:]) - 1)), label = r'$\gamma = $'+str(l5))
plt.legend()
plt.xlabel(r'$\omega_0 t$')
plt.ylabel(r'$k_B \mathcal{T}(t)/\hbar \omega_0$')

In [ ]:
# heat current and power for the sawtooth drive (lambda = l4)
plt.plot((alpha * t)/(2 * np.pi), J_l4_sw/alpha, label=r'$\mathcal{J}/\hbar\omega_0^2$')
plt.plot((alpha * t)/(2 * np.pi), p_l4_sw/alpha, label=r'$\mathcal{P}/\hbar\omega_0^2$')
plt.xlabel(r'$f t$')
plt.legend()


Alternate way --- use the same sawtooth wavefunction. Generate gradient. Interpolate into a function of the gradient, use the interpolated function in the differential equation solver. 

In [ ]:
# this one works --- inverse sawtooth
def saw_Temp(t, Te, drive_args):
    alpha, freq, amp = drive_args
    phase = np.pi/2
    return Te * (freq + amp * sy.signal.sawtooth(alpha * t + phase, width = 0.5))

Te = 1.5
l1 = 0.001  # zero coupling to the environment 
l2 = 0.00001 # weak coupling to the environment
l3 = 0.01 # somewhat stronger coupling to the environment
amp = 0.1
freq = 1

drive_args = [0.1, 1, 0.1]
t = np.arange(0, 250, 0.05)
dt = t[1] - t[0]

sawtooth_temp_profile = saw_Temp(t, Te, drive_args)           # Generating the temperature profile


#TRICKY PART ----- The gradient of the temperature, depends on the dt chosen. 
saw_dTdt = np.gradient(sawtooth_temp_profile, dt)                 

omi = 1
saw_dTdt = sy.interpolate.CubicSpline(t, saw_dTdt)            # Interpolating the gradient to get a function
saw_omega_l2 = sy.integrate.odeint(driving_protocol, omi, t, args = (saw_Temp, saw_dTdt, [l2, Te], drive_args))
saw_omega_l1 = sy.integrate.odeint(driving_protocol, omi, t, args = (saw_Temp, saw_dTdt, [l1, Te], drive_args))
saw_omega_l3 = sy.integrate.odeint(driving_protocol, omi, t, args = (saw_Temp, saw_dTdt, [l3, Te], drive_args))
saw_omega_l4 = sy.integrate.odeint(driving_protocol, omi, t, args = (saw_Temp, saw_dTdt, [l4, Te], drive_args))
saw_omega_l5 = sy.integrate.odeint(driving_protocol, omi, t, args = (saw_Temp, saw_dTdt, [l5, Te], drive_args))

# Heat current calculation
rho_sw_inv_l3 = time_evolve(saw_omega_l3[:,0], Te, saw_Temp(t[0], Te, drive_args), l3, t)
J_l3_sw_inv= dQdt(saw_omega_l3[:,0], t, l3, Te, rho_sw_inv_l3.T.reshape((len(t), 2, 2)))
p_l3_sw_inv = power(saw_omega_l3[:,0], t, rho_sw_inv_l3.T.reshape((len(t), 2, 2)))

plt.plot(t, saw_omega_l2, label = '0.00001')
plt.plot(t, saw_omega_l1, label = '0.001')
plt.plot(t, saw_omega_l3, label = '0.01')
plt.plot(t, saw_omega_l4, label = str(l4))
plt.plot(t, saw_omega_l5, label = str(l5))
#plt.plot(t, sawtooth_temp_profile)
plt.legend()

In [ ]:
#heat current and power for the sawtooth inverse (lambda = l3)
plt.plot((alpha * t)/(2 * np.pi), J_l3_sw_inv/alpha, label=r'$\mathcal{J}/\hbar\omega_0^2$')
plt.plot((alpha * t)/(2 * np.pi), p_l3_sw_inv/alpha, label=r'$\mathcal{P}/\hbar\omega_0^2$')
plt.xlabel(r'$f t$')
plt.legend()


In [ ]:
# feedback check: feed the inverted frequency back into the Lindblad
# equation and confirm we recover the sawtooth temperature profile.
rho_saw_check_l2 = time_evolve(saw_omega_l2[:, 0], Te, saw_Temp(t[0], Te, drive_args), l2, t)
rho_saw_check_l3 = time_evolve(saw_omega_l3[:, 0], Te, saw_Temp(t[0], Te, drive_args), l3, t)
rho_saw_check_l4 = time_evolve(saw_omega_l4[:, 0], Te, saw_Temp(t[0], Te, drive_args), l4, t)

plt.plot(t, saw_Temp(t, Te, drive_args), 'k--', label='target')
plt.plot(t, saw_omega_l2[:, 0]/np.log((1/rho_saw_check_l2[0, :]) - 1), label=r'$\lambda=$'+str(l2))
plt.plot(t, saw_omega_l3[:, 0]/np.log((1/rho_saw_check_l3[0, :]) - 1), label=r'$\lambda=$'+str(l3))
plt.plot(t, saw_omega_l4[:, 0]/np.log((1/rho_saw_check_l4[0, :]) - 1), label=r'$\lambda=$'+str(l4))
plt.xlabel(r'$\omega_0 t$')
plt.ylabel(r'$k_B \mathcal{T}(t)/\hbar\omega_0$')
plt.legend()


Square pulses

In [ ]:
l1 = 0.001                      # zero coupling to the environment 
l2 = 0.00001                    # weak coupling to the environment
l3 = 0.01                       # somewhat stronger coupling to the environment

freq = 1
t = np.arange(0, 250, 0.05)
p = 2 * np.pi/alpha
b = 150
#os = freq + 0.1 * sy.signal.square(alpha * t)

phase = - np.pi/2
part1 = np.sqrt((1 + b**2)/(1 + b**2 * np.cos(2 * np.pi * t/p + phase)**2)) 
os = freq + 0.1 * part1 * np.cos(2 * np.pi * t/p + phase)

rho_l1s = time_evolve(os, Te, Te, l1, t)
rho_l2s = time_evolve(os, Te, Te, l2, t)
rho_l3s = time_evolve(os, Te, Te, l3, t)
rho_l4s = time_evolve(os, Te, Te, l4, t)
rho_l5s = time_evolve(os, Te, Te, l5, t)

# Heat current calculation
J_l4_sq= dQdt(os, t, l4, Te, rho_l4s.T.reshape((len(t), 2, 2)))
p_l4_sq = power(os, t, rho_l4s.T.reshape((len(t), 2, 2)))

p_l4_sq_a = power_a(os, t, os/(freq * np.log((1/rho_l4s[0,:]) - 1)))

#plt.plot(freq * t, (freq + 0.1 * sy.signal.square(alpha * t))*Te, label = 'Theory')
plt.figure(figsize=(20, 10))
plt.plot(freq * t, os/(freq * np.log((1/rho_l2s[0,:]) - 1)), label = r'$\gamma = $'+str(l2))
plt.plot(freq * t, os/(freq * np.log((1/rho_l3s[0,:]) - 1)), label = r'$\gamma = $'+str(l3))
plt.plot(freq * t, os/(freq * np.log((1/rho_l1s[0,:]) - 1)), label = r'$\gamma = $'+str(l1))
plt.plot(freq * t, os/(freq * np.log((1/rho_l4s[0,:]) - 1)), label = r'$\gamma = $'+str(l4))
plt.plot(freq * t, os/(freq * np.log((1/rho_l5s[0,:]) - 1)), label = r'$\gamma = $'+str(l5))
plt.legend()
plt.xlabel(r'$\omega_0 t$')
plt.ylabel(r'$k_B \mathcal{T}(t)/\hbar \omega_0$')

In [ ]:
# heat current and power for the square-like drive (lambda = l4)
plt.plot((alpha * t)/(2 * np.pi), J_l4_sq/alpha, label=r'$\mathcal{J}/\hbar\omega_0^2$')
plt.plot((alpha * t)/(2 * np.pi), p_l4_sq/alpha, label=r'$\mathcal{P}/\hbar\omega_0^2$')
plt.xlabel(r'$f t$')
plt.ylim([-0.02, 0.02])
plt.legend()


In [ ]:
# the inverse
def sq_temp_prof(t, Te, drive_args):
    p, b, amp = drive_args
    phase = - np.pi/2
    part1 = np.sqrt((1 + b**2)/(1 + b**2 * np.cos(2 * np.pi * t/p + phase)**2)) 
    Temp_sq = Te + amp * part1 * np.cos(2 * np.pi * t/p + phase)
    return Temp_sq

def sq_dTdt_prof(drive_args, t):
    p, b, amp = drive_args
    phase = -np.pi/2
    part1 = np.sqrt((1 + b**2)/(1 + b**2 * np.cos(2 * np.pi * t/p + phase)**2)) 
    Temp_sq_der = - (2 * np.pi * amp * part1 * np.sin(2 * np.pi * t/p + phase))/(p + b**2 * p * np.cos(2 * np.pi * t/p + phase)**2)
    return Temp_sq_der 

t = np.arange(0, 250, 0.05)
b = 100
p = 2 * np.pi/alpha
Te = 1.5
l = 0.001

env_args = [l, Te]
drive_args = [p, b, 0.2]

temp_sq_der = ft.partial(sq_dTdt_prof, drive_args)

omi = 1
omega_sq_l1 = sy.integrate.odeint(driving_protocol, omi, t, args= (sq_temp_prof, temp_sq_der, [l1, Te], drive_args))
omega_sq_l2 = sy.integrate.odeint(driving_protocol, omi, t, args= (sq_temp_prof, temp_sq_der, [l2, Te], drive_args))
omega_sq_l3 = sy.integrate.odeint(driving_protocol, omi, t, args= (sq_temp_prof, temp_sq_der, [l3, Te], drive_args))
omega_sq_l4 = sy.integrate.odeint(driving_protocol, omi, t, args= (sq_temp_prof, temp_sq_der, [l4, Te], drive_args))
omega_sq_l5 = sy.integrate.odeint(driving_protocol, omi, t, args= (sq_temp_prof, temp_sq_der, [l5, Te], drive_args))

# Heat current calculation
rho_sq_inv_l3 = time_evolve(omega_sq_l3[:,0], Te, sq_temp_prof(t[0], Te, drive_args), l3, t)
J_l3_sq_inv= dQdt(omega_sq_l3[:,0], t, l3, Te, rho_sq_inv_l3.T.reshape((len(t), 2, 2)))
p_l3_sq_inv = power(omega_sq_l3[:,0], t, rho_sq_inv_l3.T.reshape((len(t), 2, 2)))

rho_sq_inv_l4 = time_evolve(omega_sq_l4[:,0], Te, sq_temp_prof(t[0], Te, drive_args), l4, t)
J_l4_sq_inv= dQdt(omega_sq_l4[:,0], t, l4, Te, rho_sq_inv_l4.T.reshape((len(t), 2, 2)))
p_l4_sq_inv = power(omega_sq_l4[:,0], t, rho_sq_inv_l4.T.reshape((len(t), 2, 2)))

plt.plot(t, omega_sq_l1)
plt.plot(t, omega_sq_l2)
plt.plot(t, omega_sq_l3)
plt.plot(t, omega_sq_l4)
#plt.plot(t, omega_sq_l5)   # this one is a bit much


In [ ]:
# heat current and power for the square inverse (lambda = l3)
plt.plot((alpha * t)/(2 * np.pi), J_l3_sq_inv/alpha, label=r'$\mathcal{J}/\hbar\omega_0^2$')
plt.plot((alpha * t)/(2 * np.pi), p_l3_sq_inv/alpha, label=r'$\mathcal{P}/\hbar\omega_0^2$')
plt.xlabel(r'$f t$')
plt.ylim([-0.1, 0.1])
plt.legend()


In [ ]:
# feedback check: feed the inverted frequency back into the Lindblad
# equation and confirm we recover the square temperature profile.
rho_sq_check_l2 = time_evolve(omega_sq_l2[:, 0], Te, sq_temp_prof(t[0], Te, drive_args), l2, t)
rho_sq_check_l3 = time_evolve(omega_sq_l3[:, 0], Te, sq_temp_prof(t[0], Te, drive_args), l3, t)
rho_sq_check_l4 = time_evolve(omega_sq_l4[:, 0], Te, sq_temp_prof(t[0], Te, drive_args), l4, t)

plt.plot(t, sq_temp_prof(t, Te, drive_args), 'k--', label='target')
plt.plot(t, omega_sq_l2[:, 0]/np.log((1/rho_sq_check_l2[0, :]) - 1), label=r'$\lambda=$'+str(l2))
plt.plot(t, omega_sq_l3[:, 0]/np.log((1/rho_sq_check_l3[0, :]) - 1), label=r'$\lambda=$'+str(l3))
plt.plot(t, omega_sq_l4[:, 0]/np.log((1/rho_sq_check_l4[0, :]) - 1), label=r'$\lambda=$'+str(l4))
plt.xlabel(r'$\omega_0 t$')
plt.ylabel(r'$k_B \mathcal{T}(t)/\hbar\omega_0$')
plt.legend()


The one where the qubit is driven and the temperature is left to evolve.

In [ ]:
plt.rcParams.update({'font.size': 25})
fig = plt.figure( figsize=(30, 12))

x_position_label = 2.6

gs = GridSpec(2, 6, hspace = 0, wspace = 0.5)
gs.update(left = 0.1, right = 0.95, top = 1, bottom = 0.6)

ax4 = fig.add_subplot(gs[0:, 4:6])
#ax4.plot(alpha * t, o/(Te * np.log((1/rho_l1[0,:]) - 1)), color = '#2D8077', linewidth = 7, alpha = 0.6, label = r'$\lambda = 10^{-3}$')
#ax4.plot(alpha * t, o/(Te * np.log((1/rho_l3[0,:]) - 1)), color = '#2D8077', linewidth = 4, label = r'$\lambda = 10^{-2}$')
ax4.plot((alpha * t)/(2 * np.pi), o/(Te * np.log((1/rho_l5[0,:]) - 1)), color = '#50b498', linewidth = 8, label = r'$\lambda = 5 \times 10^{-2}$')
ax4.plot((alpha * t)/(2 * np.pi), o/(Te * np.log((1/rho_l4[0,:]) - 1)), color = '#9cdba6', linewidth = 7, label = r'$\lambda = 5 \times 10^{-3}$')
ax4.plot((alpha * t)/(2 * np.pi), o/(Te * np.log((1/rho_l2[0,:]) - 1)), color = '#004e64',linewidth = 3.5, alpha = 0.8, label = r'$\lambda = 10^{-5}$')
plt.xlim([0, 2.78])
ax4.tick_params(direction='out', length=6, width=2)
plt.tick_params('y')
plt.tick_params('x', labelbottom=False)
plt.text(x_position_label, 1.09, '(c)')

ax3 = fig.add_subplot(gs[0:, 2:4])
#ax3.plot(freq * t, osw/(Te * np.log((1/rho_l1sw[0,:]) - 1)), color = '#2D8077', alpha = 0.6, linewidth = 7, label = r'$\lambda = 10^{-3}$')
#ax3.plot(freq * t, osw/(Te * np.log((1/rho_l3sw[0,:]) - 1)), color = '#2D8077', linewidth = 4, label = r'$\lambda = 10^{-2}$')
ax3.plot((alpha * t)/(2 * np.pi), osw/(Te * np.log((1/rho_l5sw[0,:]) - 1)), color = '#50b498', linewidth = 8, label = r'$\lambda = 5 \times 10^{-2}$')
ax3.plot((alpha * t)/(2 * np.pi), osw/(Te * np.log((1/rho_l4sw[0,:]) - 1)), color = '#9cdba6', linewidth = 7, label = r'$\lambda = 5 \times 10^{-3}$')
ax3.plot((alpha * t)/(2 * np.pi), osw/(Te * np.log((1/rho_l2sw[0,:]) - 1)), color = '#004e64',linewidth = 3.5, alpha = 0.8, label = r'$\lambda = 10^{-5}$')
#plt.legend(frameon = False, loc = 'upper left')
ax3.tick_params(direction='out', length=6, width=2)
plt.xlim([0, 2.78])
plt.tick_params('y')
plt.tick_params('x', labelbottom=False)
plt.text(x_position_label, 1.09, '(b)')

ax2 = fig.add_subplot(gs[0:, 0:2])
#ax2.plot(alpha * t, os/(Te * np.log((1/rho_l1s[0,:]) - 1)), color = '#2D8077', alpha = 0.6, linewidth = 7,  label = r'$\lambda =10^{-3}$')
#ax2.plot(alpha * t, os/(Te * np.log((1/rho_l3s[0,:]) - 1)), color = '#2D8077', linewidth = 4, label = r'$\lambda = 10^{-2}$')
ax2.plot((alpha * t)/(2 * np.pi), os/(Te * np.log((1/rho_l5s[0,:]) - 1)), color = '#50b498', linewidth = 8, label = r'$\lambda = 0.05$')
ax2.plot((alpha * t)/(2 * np.pi), os/(Te * np.log((1/rho_l4s[0,:]) - 1)), color = '#9cdba6', linewidth = 7, label = r'$\lambda = 0.005$')
ax2.plot((alpha * t)/(2 * np.pi), os/(Te * np.log((1/rho_l2s[0,:]) - 1)), color = '#004e64',linewidth = 3.5, alpha = 0.8, label = r'$\lambda = 10^{-5}$')
ax2.set_ylabel(r'$T(t)/T_0$')
ax2.tick_params(direction='out', length=6, width=2)
plt.xlim([0, 2.78])
plt.tick_params('x', labelbottom=False)
plt.text(x_position_label, 1.18, '(a)')
plt.legend(frameon = False, loc = 'upper left', bbox_to_anchor=(1.5, 0.45))

gs2 = GridSpec(2, 6, hspace = 0, wspace = 0.5)
gs2.update(left = 0.1, right = 0.95, top = 0.55, bottom = 0.1)
ax1 = fig.add_subplot(gs2[0:, 0:2], sharex = ax2)
ax1.plot((alpha * t)/(2 * np.pi), J_l4_sq/alpha, color = 'red', linewidth = 5, alpha = 0.7, label = r'$\mathcal{J}/\mathcal{J}_0$')
ax1.plot((alpha * t)/(2 * np.pi), p_l4_sq/alpha, color = '#DE9E46', linewidth = 5, label = r'$\mathcal{P}/\mathcal{P}_0$')
ax1.set_ylim([-0.017, 0.018])
ax1.tick_params(direction='out', length=6, width=2)
ytick_sq = np.array([-0.01, 0.000, 0.01])
ax1.set_yticks(ytick_sq, labels = [y for y in ytick_sq])
ax1.set_xlabel(r'$f t$')
#ax1.set_ylabel(r'$\mathcal{J}(t)/\hbar \omega_0^2, \mathcal{P}(t)/\hbar \omega_0^2$')
plt.locator_params(axis='x', nbins=5)
plt.text(1.25, 0.013, r'$\lambda = 5 \times 10^{-3}$')
plt.text(x_position_label, 0.013, '(d)')
plt.legend(frameon = False, loc = 'lower left')

ax12 = fig.add_subplot(gs2[0:, 2:4], sharex = ax3)
ax12.plot((alpha * t)/(2 * np.pi), J_l4_sw/alpha, color = 'red', linewidth = 5, alpha = 0.7, label = r'$\mathcal{J}(t)/\hbar \omega_0$')
ax12.plot((alpha * t)/(2 * np.pi), p_l4_sw/alpha, color = '#DE9E46', linewidth = 5, label = r'$\mathcal{P}(t)/\hbar \omega_0$')
ax12.set_xlabel(r'$f t$')
ax12.tick_params(direction='out', length=6, width=2)
ax12.set_ylim([-0.027, 0.027])
plt.text(x_position_label, 0.013, '(e)')
plt.text(1.25, 0.013, r'$\lambda = 5 \times 10^{-3}$')
plt.locator_params(axis='x', nbins=5)
ytick_saw = np.array([-0.02, -0.01, 0.000, 0.01, 0.02])
ax12.set_yticks(ytick_saw, labels = [y for y in ytick_saw])

ax13 = fig.add_subplot(gs2[0:, 4:6], sharex = ax4)
ax13.plot((alpha * t)/(2 * np.pi), J_l4_sine/alpha, color = 'red', linewidth = 5, alpha = 0.7, label = r'$\mathcal{J}(t)/\hbar \omega_0^2$')
ax13.plot((alpha * t)/(2 * np.pi), p_l4_sine/alpha, color = '#DE9E46', linewidth = 5, label = r'$\mathcal{P}(t)/\hbar \omega_0^2$')
ax13.set_xlabel(r'$f t$')
ax13.tick_params(direction='out', length=6, width=2)
plt.text(x_position_label, 0.018, '(f)')
plt.text(1.25, 0.018, r'$\lambda = 5 \times 10^{-3}$')
plt.locator_params(axis='x', nbins=5)
ytick_sine = np.array([-0.04, -0.02, 0.000, 0.02, 0.04])
ax13.set_yticks(ytick_sine, labels = [y for y in ytick_sine])
plt.ylim([-0.042,0.042])

#plt.tight_layout()
plt.savefig('1qubit_grndstate.svg', bbox_inches='tight', dpi = 150, pad_inches=0, transparent=True)
plt.show()


The one where the temperature is predermined and the frequencies are calculated. The one below is rectangular and of the same type as uploaded in the overleaf file.

In [ ]:
plt.rcParams.update({'font.size': 25})
fig = plt.figure( figsize=(30, 12))

x_position_label = 1

gs = GridSpec(2, 6, hspace = 0, wspace = 0.5)
gs.update(left = 0.1, right = 0.95, top = 1, bottom = 0.6)

# Cosine
ax4 = fig.add_subplot(gs[0:, 4:6])
#ax4.plot(freq * t, osine_inv_l1, color = '#2D8077', linewidth = 7, alpha = 0.6, label = r'$\lambda = 10^{-3}$')
ax4.plot((alpha * t)/(2 * np.pi), osine_inv_l3, color = '#50b498', linewidth = 8, label = r'$\lambda = 10^{-2}$')
ax4.plot((alpha * t)/(2 * np.pi), osine_inv_l4, color = '#9cdba6', linewidth = 7, label = r'$\lambda = 5 \times 10^{-3}$')
ax4.plot((alpha * t)/(2 * np.pi), osine_inv_l2, color = '#004e64',linewidth = 3.5, alpha = 0.8, label = r'$\lambda = 10^{-5}$')
#ax4.plot(freq * t, osine_inv_l5, color = '#2D8077', linewidth = 7, alpha = 0.6, label = r'$\lambda = 5 \times 10^{-2}$')
#ax4.set_yticks(np.array([0.9, 1, 1.1, 1.2, 1.3]))
ax4.tick_params(direction='out', length=6, width=2)
plt.ylim([0.88, 1.29])
#plt.legend(frameon = False, loc = 'upper left')
plt.xlim([0, 2.78])
#plt.tick_params('y', labelleft=False)
plt.tick_params('x', labelbottom=False)
plt.text(x_position_label,  1.09, '(e)')

# sawtooth
ax3 = fig.add_subplot(gs[0:, 2:4])
#ax3.plot(freq * t, saw_omega_l1, color = '#2D8077', alpha = 0.6, linewidth = 7, label = r'$\lambda = 10^{-3}$')
ax3.plot((alpha * t)/(2 * np.pi), saw_omega_l3, color = '#50b498', linewidth = 8, label = r'$\lambda = 10^{-2}$')
ax3.plot((alpha * t)/(2 * np.pi), saw_omega_l4, color = '#9cdba6', linewidth = 7, label = r'$\lambda = 5 \times 10^{-3}$')
ax3.plot((alpha * t)/(2 * np.pi), saw_omega_l2, color = '#004e64', alpha = 0.8, linewidth = 3.5, label = r'$\lambda = 10^{-5}$')
#ax3.plot(freq * t, saw_omega_l5, color = '#2D8077', alpha = 0.6, linewidth = 7, label = r'$\lambda = 5 \times 10^{-2}$')
#plt.legend(frameon = False, loc = 'upper left')
ax3.tick_params(direction='out', length=6, width=2)
plt.xlim([0, 2.78])
plt.ylim([0.88, 1.25])
#plt.tick_params('y', labelleft=False)
plt.tick_params('x', labelbottom=False)
plt.text(x_position_label, 1.23, '(c)')

#square
ax2 = fig.add_subplot(gs[0:, 0:2])
#ax2.plot(freq * t, omega_sq_l1, color = '#2D8077', alpha = 0.6, linewidth = 7,  label = r'$\lambda =10^{-3}$')
ax2.plot((alpha * t)/(2 * np.pi), omega_sq_l3, color = '#50b498', linewidth = 8, label = r'$\lambda = 10^{-2}$')
ax2.plot((alpha * t)/(2 * np.pi), omega_sq_l4, color = '#9cdba6', linewidth = 7, label = r'$\lambda = 5 \times 10^{-3}$')
ax2.plot((alpha * t)/(2 * np.pi), omega_sq_l2, color = '#004e64', alpha = 0.8, linewidth = 3.5, label = r'$\lambda = 10^{-5}$')
#ax2.plot(freq * t, omega_sq_l5, color = '#2D8077', alpha = 0.6, linewidth = 7,  label = r'$\lambda = 5 \times 10^{-2}$')
ax2.set_ylabel(r'$\omega(t)/\omega_0$')
#plt.locator_params(axis='y', nbins=4)
ax2.tick_params(direction='out', length=6, width=2)
ax2.set_yticks(np.array([1, 1.2, 1.4, 1.6]))
plt.xlim([0, 2.78])
plt.ylim([0.82, 1.75])
plt.tick_params('x', labelbottom=False)
plt.text(x_position_label, 1.6, '(a)')
plt.legend(frameon = False, loc = 'upper left', bbox_to_anchor=(0.17, 1))

gs2 = GridSpec(2, 6, hspace = 0, wspace = 0.5)
gs2.update(left = 0.1, right = 0.95, top = 0.55, bottom = 0.1)

#square heat current/power
ax1 = fig.add_subplot(gs2[0:, 0:2], sharex = ax2)
ax1.plot((alpha * t)/(2 * np.pi), J_l3_sq_inv/alpha, color = 'red', linewidth = 5, alpha = 0.7, label = r'$\mathcal{J}$')
ax1.plot((alpha * t)/(2 * np.pi), p_l3_sq_inv/alpha, color = '#DE9E46', linewidth = 5, label = r'$\mathcal{P}$')
ax1.set_ylim([-0.09, 0.09])
ytick_sq = np.array([-0.08, -0.04, 0.000, 0.04, 0.08])
ax1.tick_params(direction='out', length=6, width=2)
ax1.set_yticks(ytick_sq, labels = [y for y in ytick_sq])
ax1.set_xlabel(r'$f t$')
#ax1.set_ylabel(r'$\mathcal{J}(t)/\hbar \omega_0^2, \mathcal{P}(t)/\hbar \omega_0^2 \ \ \ \times 10^{-3}$')
plt.locator_params(axis='x', nbins=10)
yposition_ax1 = 0.078
plt.text(1.5, yposition_ax1, r'$\lambda = 10^{-2}$')
plt.text(x_position_label, yposition_ax1, '(b)')
plt.legend(frameon = False, loc = 'lower left')

#sawtooth heat current/power
ax12 = fig.add_subplot(gs2[0:, 2:4], sharex = ax3)
ax12.plot((alpha * t)/(2 * np.pi), J_l3_sw_inv/alpha, color = 'red', linewidth = 5, alpha = 0.7, label = r'$\mathcal{J}(t)/\hbar \omega_0$')
ax12.plot((alpha * t)/(2 * np.pi), p_l3_sw_inv/alpha, color = '#DE9E46', linewidth = 5, label = r'$\mathcal{P}(t)/\hbar \omega_0$')
ax12.set_xlabel(r'$f t$')
ax12.set_ylim([-0.058, 0.065])
ax12.tick_params(direction='out', length=6, width=2)
ytick_saw = np.array([-0.04, 0.00, 0.04])
ax12.set_yticks(ytick_saw, labels = [y for y in ytick_saw])
yposition_ax12 = 0.039
plt.text(x_position_label, yposition_ax12, '(d)')
plt.text(1.5, yposition_ax12, r'$\lambda = 10^{-2}$')
#plt.tick_params('y', labelleft=False)
#plt.legend(frameon = False, loc = 'lower left')

#sine heat current/power
ax13 = fig.add_subplot(gs2[0:, 4:6], sharex = ax3)
ax13.plot((alpha * t)/(2 * np.pi), J_l3_sine_inv/alpha, color = 'red', linewidth = 5, alpha = 0.7, label = r'$\mathcal{J}(t)/\hbar \omega_0^2$')
ax13.plot((alpha * t)/(2 * np.pi), p_l3_sine_inv/alpha, color = '#DE9E46', linewidth = 5, label = r'$\mathcal{P}(t)/\hbar \omega_0^2$')
ax13.set_xlabel(r'$f t$')
ax13.tick_params(direction='out', length=6, width=2)
ytick_sine = np.array([-0.04, 0.00, 0.04])
ax13.set_yticks(ytick_sine, labels = [y for y in ytick_sine])
ax13.set_ylim([-0.055, 0.055])
plt.locator_params(axis='x', nbins=10)
plt.text(x_position_label, yposition_ax12, '(f)')
plt.text(1.5, yposition_ax12, r'$\lambda = 10^{-2}$')
#plt.tick_params('y', labelleft=False)
#plt.legend(frameon = False, loc = 'lower left')

#plt.tight_layout()
plt.savefig('1qubit_inverse_grndstate.svg', bbox_inches='tight', dpi = 150, pad_inches=0, transparent=True)
plt.show()
